# N04 - Development CV, model selection, locked test (plan stage 4, sections 5.2 and 6)
Gate: candidate log complete; LR / tree winner / MLP selected **without** any test-based tuning.

**Same records for every model variant**
- Every configuration of every model family is trained on the *same* fold arrays made in N03 (same rows, same preprocessing). A fingerprint (hash) of the training and validation row ids is logged for every run and checked at the end.
- **Epoch rule for the MLP:** all MLP variants use the *same fitting rows* (training rows minus the frozen early-stopping rows) and the *same record order in every epoch*: epoch `e` always uses the permutation `get_epoch_order(n_rows, e, seed)`, which depends only on the epoch number and the data-order seed - not on layers, learning rate or weight decay. Each epoch uses every fitting record exactly once (asserted).
- Early stopping uses the frozen inner PSU-disjoint early-stopping rows, **never** the scoring fold.

Selection rule (plan 6.1): best mean fold AP; ties within a pre-declared tolerance -> lower mean log loss -> simpler configuration. The test partition is opened once, at the end.

In [ ]:
# --- setup (same in every notebook) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab - using local folder.")
import os, sys, time
import numpy as np
import pandas as pd
BASE_DIR = os.environ.get("THESIS_BASE_DIR", "/content/drive/MyDrive/THESIS_V2")
sys.path.append(BASE_DIR + "/code")
import bdhs_helpers as h
h.make_all_folders()
h.set_all_seeds(h.SEED_SPLIT)
h.print_environment("N04")

import joblib
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
print("TIE_TOLERANCE (AP):", h.TIE_TOLERANCE, "| LOGLOSS_TIE:", h.LOGLOSS_TIE)

In [ ]:
h.require_signoff("n02_signoff.txt", "finish N02")
if not os.path.exists(h.OUTPUT_DIR + "/n03_leakage_checks.csv"):
    raise RuntimeError("Run N03 first.")

# load the frozen fold arrays (identical for every model)
train_ids_list = []
val_ids_list = []
X_train_list = []
X_val_list = []
Xs_train_list = []
Xs_val_list = []
y_train_list = []
y_val_list = []
es_list = []
for k in range(h.N_DEV_FOLDS):
    data = np.load(h.PRIVATE_DIR + "/fold" + str(k) + "_arrays.npz")
    train_ids_list.append(data["train_ids"])
    val_ids_list.append(data["val_ids"])
    X_train_list.append(data["X_train"])
    X_val_list.append(data["X_val"])
    Xs_train_list.append(data["Xs_train"])
    Xs_val_list.append(data["Xs_val"])
    y_train_list.append(data["y_train"])
    y_val_list.append(data["y_val"])
    es_list.append(data["es_mask"])
    print("fold", k, "| train", X_train_list[k].shape, "| val", X_val_list[k].shape, "| ES rows", int(es_list[k].sum()))

### Runtime pilot (development data only)
Times one configuration per family on fold 0, then estimates the full search time. Use it to decide if the search fits Colab. If you shrink the grid, write the change in the plan first.

In [ ]:
k = 0
t = time.time()
LogisticRegression(C=1.0, max_iter=2000).fit(Xs_train_list[k], y_train_list[k])
sec_lr = time.time() - t

t = time.time()
RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=h.RF_N_JOBS).fit(X_train_list[k], y_train_list[k])
sec_rf = time.time() - t

t = time.time()
XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=200, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=h.XGB_N_JOBS, eval_metric="logloss").fit(X_train_list[k], y_train_list[k])
sec_xgb = time.time() - t

t = time.time()
mask = ~es_list[k]
h.train_mlp(Xs_train_list[k][mask], y_train_list[k][mask], Xs_train_list[k][~mask], y_train_list[k][~mask], [32, 16], 0.001, 0.0, h.MLP_DATA_ORDER_SEED + k)
sec_mlp = time.time() - t

print("seconds per fit: LR", round(sec_lr, 2), "| RF", round(sec_rf, 2), "| XGB", round(sec_xgb, 2), "| MLP", round(sec_mlp, 2))
total = 4 * 5 * sec_lr + 9 * 5 * sec_rf + 12 * 5 * sec_xgb + 12 * 5 * sec_mlp
print("rough full-search estimate:", round(total / 60.0, 1), "minutes")

### Search log
Columns of the log: `p1..p4` hold the configuration numbers (legend):
- **LR:** p1 = C
- **RF:** p1 = max_depth (0 means None), p2 = min_samples_leaf
- **XGB:** p1 = max_depth, p2 = learning_rate, p3 = n_estimators
- **MLP:** p1 = first hidden size, p2 = second hidden size (0 = none), p3 = learning_rate, p4 = weight_decay

`complexity` is used only for the final tie-break (smaller = simpler).

In [ ]:
LOG_COLUMNS = ["model", "config", "p1", "p2", "p3", "p4", "complexity", "fold", "n_train", "n_val",
               "train_ids_hash", "val_ids_hash", "ap", "roc_auc", "log_loss", "brier", "seconds", "seed",
               "best_epoch", "epochs_run", "order_hash_10", "fit_ids_hash", "es_ids_hash"]
log_rows = []
oof_names = []       # "MODEL|config" for every configuration
oof_preds = []       # for each configuration: list of validation predictions, one array per fold


def add_log_row(model, config, p1, p2, p3, p4, complexity, fold, probs, seconds, seed,
                best_epoch, epochs_run, order_hash, fit_hash, es_hash):
    ap, auc, ll, brier = h.score_probs(y_val_list[fold], probs)
    log_rows.append([model, config, p1, p2, p3, p4, complexity, fold,
                     len(train_ids_list[fold]), len(val_ids_list[fold]),
                     h.ids_hash(train_ids_list[fold]), h.ids_hash(val_ids_list[fold]),
                     ap, auc, ll, brier, seconds, seed, best_epoch, epochs_run, order_hash, fit_hash, es_hash])
    print(model, config, "fold", fold, "AP", round(ap, 4), "| log loss", round(ll, 4), "|", round(seconds, 1), "s")

In [ ]:
# ---------- Logistic Regression (L2 is the scikit-learn default) ----------
for C in [0.01, 0.1, 1, 10]:
    config_text = "C=" + str(C)
    fold_preds = []
    for k in range(h.N_DEV_FOLDS):
        start = time.time()
        model = LogisticRegression(C=C, solver="lbfgs", max_iter=2000)
        model.fit(Xs_train_list[k], y_train_list[k])
        if list(model.classes_) != [0, 1]:
            raise AssertionError("class order is not [0, 1]")
        probs = model.predict_proba(Xs_val_list[k])[:, 1]
        fold_preds.append(probs)
        add_log_row("LR", config_text, C, 0, 0, 0, C, k, probs, time.time() - start, -1, -1, -1, "", "", "")
    oof_names.append("LR|" + config_text)
    oof_preds.append(fold_preds)

In [ ]:
# ---------- Random Forest (n_estimators = 300 fixed, random_state = 42, n_jobs recorded) ----------
for depth in [5, 10, 0]:                 # 0 stands for max_depth = None
    for leaf in [5, 10, 20]:
        config_text = "depth=" + str(depth) + ",leaf=" + str(leaf)
        depth_for_model = None if depth == 0 else depth
        depth_number = 50 if depth == 0 else depth
        complexity = depth_number / leaf
        fold_preds = []
        for k in range(h.N_DEV_FOLDS):
            start = time.time()
            model = RandomForestClassifier(n_estimators=300, max_depth=depth_for_model, min_samples_leaf=leaf,
                                           random_state=42, n_jobs=h.RF_N_JOBS)
            model.fit(X_train_list[k], y_train_list[k])
            if list(model.classes_) != [0, 1]:
                raise AssertionError("class order is not [0, 1]")
            probs = model.predict_proba(X_val_list[k])[:, 1]
            fold_preds.append(probs)
            add_log_row("RF", config_text, depth, leaf, 300, 0, complexity, k, probs, time.time() - start, 42, -1, -1, "", "", "")
        oof_names.append("RF|" + config_text)
        oof_preds.append(fold_preds)

In [ ]:
# ---------- XGBoost (subsample = colsample = 0.8, reg_lambda = 1, min_child_weight = 1, random_state = 42) ----------
for depth in [2, 3, 4]:
    for lr in [0.05, 0.1]:
        for n_est in [100, 200]:
            config_text = "depth=" + str(depth) + ",lr=" + str(lr) + ",n=" + str(n_est)
            fold_preds = []
            for k in range(h.N_DEV_FOLDS):
                start = time.time()
                model = XGBClassifier(max_depth=depth, learning_rate=lr, n_estimators=n_est,
                                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=1,
                                      objective="binary:logistic", eval_metric="logloss",
                                      tree_method="hist", random_state=42, n_jobs=h.XGB_N_JOBS)
                model.fit(X_train_list[k], y_train_list[k])
                probs = model.predict_proba(X_val_list[k])[:, 1]
                fold_preds.append(probs)
                add_log_row("XGB", config_text, depth, lr, n_est, 0, depth * n_est, k, probs, time.time() - start, 42, -1, -1, "", "", "")
            oof_names.append("XGB|" + config_text)
            oof_preds.append(fold_preds)

### MLP search - same records, same order in every epoch, for every variant

In [ ]:
# ---------- MLP ----------
# Every variant (any layers / learning rate / weight decay) gets:
#   * the same fitting rows   = training rows minus the frozen early-stopping rows
#   * the same record order   = get_epoch_order(n_fit, epoch, MLP_DATA_ORDER_SEED + fold) in epoch 0, 1, 2, ...
#   * the same initial-weight seed (MLP_INIT_SEED) and the same early-stopping rows
for hidden_text in ["32", "64", "32-16"]:
    if hidden_text == "32":
        hidden_sizes = [32]
        hidden_1 = 32
        hidden_2 = 0
    elif hidden_text == "64":
        hidden_sizes = [64]
        hidden_1 = 64
        hidden_2 = 0
    else:
        hidden_sizes = [32, 16]
        hidden_1 = 32
        hidden_2 = 16
    for lr in [0.001, 0.0005]:
        for wd in [0.0, 0.001]:
            config_text = "hidden=" + hidden_text + ",lr=" + str(lr) + ",wd=" + str(wd)
            fold_preds = []
            for k in range(h.N_DEV_FOLDS):
                start = time.time()
                use_es = es_list[k]
                X_fit = Xs_train_list[k][~use_es]
                y_fit = y_train_list[k][~use_es]
                X_es = Xs_train_list[k][use_es]
                y_es = y_train_list[k][use_es]
                order_seed = h.MLP_DATA_ORDER_SEED + k
                model, best_epoch, epochs_run, order_hash, n_params = h.train_mlp(
                    X_fit, y_fit, X_es, y_es, hidden_sizes, lr, wd, order_seed)
                probs = h.predict_mlp(model, Xs_val_list[k])
                fold_preds.append(probs)
                fit_hash = h.ids_hash(train_ids_list[k][~use_es])
                es_hash = h.ids_hash(train_ids_list[k][use_es])
                add_log_row("MLP", config_text, hidden_1, hidden_2, lr, wd, n_params, k, probs,
                            time.time() - start, order_seed, best_epoch, epochs_run, order_hash, fit_hash, es_hash)
            oof_names.append("MLP|" + config_text)
            oof_preds.append(fold_preds)

### Check: every model variant saw the same records
If any line below says FAIL the run stops. The audit is saved as `n04_record_audit.csv`.

In [ ]:
log_df = pd.DataFrame(log_rows, columns=LOG_COLUMNS)
h.save_table(log_df, h.OUTPUT_DIR + "/T4_tuning_log_all_configs_folds.csv")

check_log = []
audit_rows = []
for k in range(h.N_DEV_FOLDS):
    part = log_df[log_df["fold"] == k]
    expected_train = h.ids_hash(train_ids_list[k])
    expected_val = h.ids_hash(val_ids_list[k])
    n_runs = len(part)
    all_same_train = bool((part["train_ids_hash"] == expected_train).all())
    all_same_val = bool((part["val_ids_hash"] == expected_val).all())
    h.add_check(check_log, "fold " + str(k) + ": all " + str(n_runs) + " runs (LR/RF/XGB/MLP) used the same training rows", all_same_train, expected_train)
    h.add_check(check_log, "fold " + str(k) + ": all runs were scored on the same validation rows", all_same_val, expected_val)

    mlp_part = part[part["model"] == "MLP"]
    n_fit_hash = mlp_part["fit_ids_hash"].nunique()
    n_es_hash = mlp_part["es_ids_hash"].nunique()
    n_order_hash = mlp_part["order_hash_10"].nunique()
    h.add_check(check_log, "fold " + str(k) + ": all " + str(len(mlp_part)) + " MLP variants used the same fitting rows", n_fit_hash == 1, "distinct fit hashes: " + str(n_fit_hash))
    h.add_check(check_log, "fold " + str(k) + ": all MLP variants used the same early-stopping rows", n_es_hash == 1, "distinct ES hashes: " + str(n_es_hash))
    h.add_check(check_log, "fold " + str(k) + ": all MLP variants used the same record order in epochs 1-10", n_order_hash == 1, "distinct order hashes: " + str(n_order_hash))
    audit_rows.append([k, n_runs, len(mlp_part), expected_train, expected_val,
                       mlp_part["fit_ids_hash"].iloc[0], mlp_part["es_ids_hash"].iloc[0], mlp_part["order_hash_10"].iloc[0],
                       int(mlp_part["epochs_run"].min()), int(mlp_part["epochs_run"].max())])
audit_df = pd.DataFrame(audit_rows, columns=["fold", "n_runs", "n_mlp_runs", "train_ids_hash", "val_ids_hash",
                                             "mlp_fit_ids_hash", "mlp_es_ids_hash", "mlp_order_hash_epochs_1_10",
                                             "min_epochs_run", "max_epochs_run"])
print(audit_df.to_string(index=False))
h.save_table(audit_df, h.OUTPUT_DIR + "/n04_record_audit.csv")

### Leakage canary (label-shuffle test)
Train logistic regression on **shuffled training labels** and score on the real validation labels. With no leakage, AP should be about the prevalence. A clearly higher AP means something leaks.

In [ ]:
canary_aps = []
prevalences = []
for k in range(h.N_DEV_FOLDS):
    shuffler = np.random.RandomState(7)
    y_shuffled = shuffler.permutation(y_train_list[k])
    model = LogisticRegression(C=1.0, max_iter=2000).fit(Xs_train_list[k], y_shuffled)
    probs = model.predict_proba(Xs_val_list[k])[:, 1]
    ap, auc, ll, brier = h.score_probs(y_val_list[k], probs)
    canary_aps.append(ap)
    prevalences.append(float(y_val_list[k].mean()))
mean_canary = float(np.mean(canary_aps))
mean_prev = float(np.mean(prevalences))
print("shuffled-label AP:", round(mean_canary, 4), "| prevalence:", round(mean_prev, 4))
h.add_check(check_log, "label-shuffle canary: AP near prevalence", abs(mean_canary - mean_prev) < 0.05,
            "AP " + str(round(mean_canary, 4)) + " vs prevalence " + str(round(mean_prev, 4)))

### Per-configuration summary and selection (development CV only)
Mean and SD are across the five validation-fold scores (SD uses n-1). Pooled AP is **not** used.

In [ ]:
unique_cfg = log_df[["model", "config", "p1", "p2", "p3", "p4", "complexity"]].drop_duplicates(subset=["model", "config"])
summary_rows = []
for i in range(len(unique_cfg)):
    m = unique_cfg["model"].iloc[i]
    c = unique_cfg["config"].iloc[i]
    part = log_df[(log_df["model"] == m) & (log_df["config"] == c)]
    summary_rows.append([m, c, unique_cfg["p1"].iloc[i], unique_cfg["p2"].iloc[i], unique_cfg["p3"].iloc[i], unique_cfg["p4"].iloc[i],
                         unique_cfg["complexity"].iloc[i], len(part),
                         part["ap"].mean(), part["ap"].std(ddof=1),
                         part["roc_auc"].mean(), part["roc_auc"].std(ddof=1),
                         part["log_loss"].mean(), part["log_loss"].std(ddof=1),
                         part["brier"].mean(), part["brier"].std(ddof=1),
                         part["seconds"].sum()])
summary = pd.DataFrame(summary_rows, columns=["model", "config", "p1", "p2", "p3", "p4", "complexity", "n_folds",
                                              "mean_ap", "sd_ap", "mean_auc", "sd_auc", "mean_logloss", "sd_logloss",
                                              "mean_brier", "sd_brier", "total_seconds"])

summary["selected_in_family"] = False
summary["selection_note"] = ""
for family in ["LR", "RF", "XGB", "MLP"]:
    winner, reason = h.select_best_config(summary, family, h.TIE_TOLERANCE, h.LOGLOSS_TIE)
    summary.loc[winner, "selected_in_family"] = True
    summary.loc[winner, "selection_note"] = reason

# tree slot: RF vs XGBoost, development CV only (plan 6.1)
rf_row = summary[(summary["model"] == "RF") & (summary["selected_in_family"])].iloc[0]
xgb_row = summary[(summary["model"] == "XGB") & (summary["selected_in_family"])].iloc[0]
if abs(rf_row["mean_ap"] - xgb_row["mean_ap"]) <= h.TIE_TOLERANCE:
    tree_winner = "RF" if rf_row["mean_logloss"] <= xgb_row["mean_logloss"] else "XGB"
    tree_reason = "AP within tolerance -> lower mean log loss"
else:
    tree_winner = "RF" if rf_row["mean_ap"] > xgb_row["mean_ap"] else "XGB"
    tree_reason = "higher mean development-fold AP"
print("Tree slot winner:", tree_winner, "(", tree_reason, ")")
summary["in_final_three"] = False
for i in summary.index:
    if summary.loc[i, "selected_in_family"]:
        fam = summary.loc[i, "model"]
        if fam == "LR" or fam == "MLP" or fam == tree_winner:
            summary.loc[i, "in_final_three"] = True

h.save_table(summary, h.OUTPUT_DIR + "/T4_config_summary.csv")
print(summary[summary["selected_in_family"]][["model", "config", "mean_ap", "sd_ap", "mean_logloss", "in_final_three", "selection_note"]].round(4).to_string(index=False))

### Threshold rule (decision D15)
Threshold-dependent metrics use the threshold that maximises F1 on the pooled **development** out-of-fold predictions of the selected configuration. The test set plays no part in it.

In [ ]:
selected = summary[summary["selected_in_family"]].reset_index(drop=True)
thr_rows = []
for i in range(len(selected)):
    name = selected["model"].iloc[i] + "|" + selected["config"].iloc[i]
    position = oof_names.index(name)
    pooled_y = np.concatenate(y_val_list)
    pooled_p = np.concatenate(oof_preds[position])
    threshold, f1_dev = h.best_f1_threshold(pooled_y, pooled_p)
    thr_rows.append([selected["model"].iloc[i], round(threshold, 2), round(f1_dev, 4)])
thr_df = pd.DataFrame(thr_rows, columns=["model", "threshold", "dev_oof_f1"])
print(thr_df.to_string(index=False))
h.save_table(thr_df, h.OUTPUT_DIR + "/n04_thresholds.csv")
selected.to_csv(h.OUTPUT_DIR + "/n04_selected_models.csv", index=False)

### LOCKED TEST EVALUATION - run once
Everything above is frozen. This cell refits the four selected configurations on the **full development partition** (the MLP again uses the frozen early-stopping rows and the same fixed record-order rule), transforms the test rows with the **development-fitted** preprocessing, and scores them once.

A flag file is written afterwards. If it exists, the cell refuses to run again (set `ALLOW_RERUN = True` only if you must, and report that the test set was opened more than once).

In [ ]:
ALLOW_RERUN = False
flag_path = h.OUTPUT_DIR + "/n04_test_opened.flag"
if os.path.exists(flag_path) and not ALLOW_RERUN:
    raise RuntimeError("The test set was already opened (flag file exists). Do not re-evaluate.")

# --- final development data (frozen in N03) ---
dev = np.load(h.PRIVATE_DIR + "/final_dev_arrays.npz")
X_dev = dev["X_dev"]
Xs_dev = dev["Xs_dev"]
y_dev = dev["y_dev"]
dev_ids = dev["dev_ids"]
es_dev = dev["es_mask"]
dev_feature_names = list(dev["feature_names"])

# --- test rows, transformed with the development-fitted preprocessing ---
frozen = pd.read_csv(h.OUTPUT_DIR + "/primary_predictors.csv")
predictors = []
for i in range(len(frozen)):
    if frozen["status"].iloc[i] == "primary":
        predictors.append(frozen["variable"].iloc[i])
cohort = pd.read_csv(h.PRIVATE_DIR + "/cohort_private.csv")
manifest = pd.read_csv(h.PRIVATE_DIR + "/split_manifest_private.csv")
if not np.array_equal(cohort["row_id"].values, manifest["row_id"].values):
    raise AssertionError("cohort and manifest rows do not line up")
if h.file_sha256(h.PRIVATE_DIR + "/split_manifest_private.csv") != open(h.OUTPUT_DIR + "/split_manifest_checksum.txt").read().strip():
    raise AssertionError("split manifest changed since N03")
test_mask = (manifest["partition"] == "test").values
test_cohort = cohort[test_mask]
spec_final = h.load_spec(h.PRIVATE_DIR + "/preproc_spec_final.csv")
test_clean = h.clean_predictors(test_cohort, predictors)
X_test_df, Xs_test_df = h.apply_preprocessing(test_clean, spec_final, predictors)
X_test = X_test_df.values
Xs_test = Xs_test_df.values
y_test = test_cohort["stunted"].values.astype(int)

final_checks = []
h.add_check(final_checks, "test columns equal development columns", list(X_test_df.columns) == dev_feature_names, str(len(dev_feature_names)) + " columns")
h.add_check(final_checks, "no PSU in both development and test", len(set(cohort["v021"].values[test_mask]) & set(cohort["v021"].values[~test_mask])) == 0, "checked again before scoring")
h.add_check(final_checks, "no test row id inside the development arrays", len(set(test_cohort["row_id"].values) & set(dev_ids)) == 0, "checked again before scoring")
print("test rows:", len(y_test), "| stunted:", int(y_test.sum()))

In [ ]:
final_rows = []
tree_name = tree_winner
for i in range(len(selected)):
    family = selected["model"].iloc[i]
    config = selected["config"].iloc[i]
    p1 = selected["p1"].iloc[i]
    p2 = selected["p2"].iloc[i]
    p3 = selected["p3"].iloc[i]
    p4 = selected["p4"].iloc[i]
    start = time.time()

    if family == "LR":
        model = LogisticRegression(C=float(p1), solver="lbfgs", max_iter=2000).fit(Xs_dev, y_dev)
        probs = model.predict_proba(Xs_test)[:, 1]
        joblib.dump(model, h.MODEL_DIR + "/final_LR.joblib")
    elif family == "RF":
        depth_for_model = None if int(p1) == 0 else int(p1)
        model = RandomForestClassifier(n_estimators=300, max_depth=depth_for_model, min_samples_leaf=int(p2),
                                       random_state=42, n_jobs=h.RF_N_JOBS).fit(X_dev, y_dev)
        probs = model.predict_proba(X_test)[:, 1]
        joblib.dump(model, h.MODEL_DIR + "/final_RF.joblib")
    elif family == "XGB":
        model = XGBClassifier(max_depth=int(p1), learning_rate=float(p2), n_estimators=int(p3),
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=1,
                              objective="binary:logistic", eval_metric="logloss",
                              tree_method="hist", random_state=42, n_jobs=h.XGB_N_JOBS).fit(X_dev, y_dev)
        probs = model.predict_proba(X_test)[:, 1]
        joblib.dump(model, h.MODEL_DIR + "/final_XGB.joblib")
    else:
        if int(p2) == 0:
            hidden_sizes = [int(p1)]
        else:
            hidden_sizes = [int(p1), int(p2)]
        mask = ~es_dev
        model, best_epoch, epochs_run, order_hash, n_params = h.train_mlp(
            Xs_dev[mask], y_dev[mask], Xs_dev[~mask], y_dev[~mask], hidden_sizes, float(p3), float(p4),
            h.MLP_DATA_ORDER_SEED + 100)
        probs = h.predict_mlp(model, Xs_test)
        torch.save(model.state_dict(), h.MODEL_DIR + "/final_MLP_state.pt")
        print("final MLP best epoch:", best_epoch, "| epochs run:", epochs_run)

    threshold = float(thr_df[thr_df["model"] == family]["threshold"].iloc[0])
    ap, auc, ll, brier = h.score_probs(y_test, probs)
    precision, recall, f1, bal = h.threshold_metrics(y_test, probs, threshold)
    in_final = bool(selected["in_final_three"].iloc[i])
    final_rows.append([family, config, in_final, threshold, len(y_test), int(y_test.sum()),
                       ap, auc, ll, brier, precision, recall, f1, bal])
    print(family, "| test AP", round(ap, 4), "| AUC", round(auc, 4), "| log loss", round(ll, 4), "|", round(time.time() - start, 1), "s")

final_df = pd.DataFrame(final_rows, columns=["model", "config", "in_final_three", "threshold_from_dev", "n_test", "n_test_stunted",
                                             "ap", "roc_auc", "log_loss", "brier", "precision", "recall", "f1", "balanced_accuracy"])
h.save_table(final_df, h.OUTPUT_DIR + "/T4_final_test_metrics.csv")
print(final_df.round(4).to_string(index=False))

f = open(flag_path, "w")
f.write("test set opened once. tree winner = " + tree_name + "\n")
f.close()
print("Test flag written. Do not re-run this cell.")

In [ ]:
check_df = pd.DataFrame(check_log + final_checks, columns=["check", "result", "detail"])
h.save_table(check_df, h.OUTPUT_DIR + "/n04_leakage_and_record_checks.csv")
print("Checks passed:", int((check_df["result"] == "PASS").sum()), "of", len(check_df))
print()
print("FINAL THREE MODELS FOR THE EXPLANATION STAGE:")
print(final_df[final_df["in_final_three"]][["model", "config"]].to_string(index=False))
print("Gate: candidate log complete, selection made on development CV only. Next: N05 (explainer pilot).")